####Schema Evolution

In [0]:
DROP DATABASE IF EXISTS demodb CASCADE;
CREATE DATABASE IF NOT EXISTS demodb;
USE demodb;


In [0]:
SELECT id, fname, lname FROM json.`/Volumes/workspace/default/data/schema/people.json`

In [0]:
DROP TABLE IF EXISTS people;

CREATE OR REPLACE TABLE people(
  id INT,
  firstName STRING,
  lastName STRING
) USING DELTA;

INSERT INTO people
SELECT id, fname, lname FROM json.`/Volumes/workspace/default/data/schema/people.json`;

In [0]:
SELECT * FROM people

In [0]:
%python
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "false") 

#####Schema Validations Summary
- `INSERT`
  - Column matching by position, New columns not allowed
- `OVERWRITE`
  - Column matching by position, New columns not allowed
- `MERGE .. INSERT`
  - Column matching by name, New columns ignored
- `DataFrame Append`
  - Column matching by name, New columns not allowed
- `Data Type Mismatch`
  - Not allowed in any case

#####Schema evolution approaches
- `Manual` - New columns
- `Automatic` - New columns

####Manual Schema Evolution

#####1. Manual schema evolution - New column at the end

In [0]:
ALTER TABLE people ADD COLUMNS (birthDate STRING);

In [0]:
DESC people

In [0]:
INSERT INTO people
SELECT id, fname, lname, dob
FROM json.`/Volumes/workspace/default/data/schema/people.json`

In [0]:
INSERT INTO people
SELECT id, fname firstName, lname lastName
FROM json.`/Volumes/workspace/default/data/schema/people.json`

In [0]:
INSERT INTO people
SELECT id, fname firstName, lname lastName, dob birthDate
FROM json.`/Volumes/workspace/default/data/schema/people.json`

In [0]:
INSERT INTO people
SELECT id, fname firstName, lname lastName, dob birthDate, current_date() toDay
FROM json.`/Volumes/workspace/default/data/schema/people.json`

In [0]:
INSERT OVERWRITE people
SELECT id, fname firstName, lname lastName
FROM json.`/Volumes/workspace/default/data/schema/people_2.json`

#####2. Manual schema evolution - New column in the middle

In [0]:
ALTER TABLE people ADD COLUMNS (phoneNumber STRING after lastName);

In [0]:
DESC people

In [0]:
INSERT INTO people
SELECT id, fname firstName, lname lastName, phone phoneNumber, dob birthDate
FROM json.`/Volumes/workspace/default/data/schema/people_2.json`

In [0]:
select * from people

####Automatic Schema Evolution - Session Level

In [0]:
DROP TABLE IF EXISTS people;

CREATE OR REPLACE TABLE people(
  id INT,
  firstName STRING,
  lastName STRING
) USING DELTA;

INSERT INTO people
SELECT id, fname, lname FROM json.`/Volumes/workspace/default/data/schema/people.json`;

SELECT * FROM people;

In [0]:
SET spark.databricks.delta.schema.autoMerge.enabled = true

#####3. Automatic schema evolution - New column at the end

In [0]:
INSERT INTO people
SELECT id, fname firstName, lname lastName, dob birthDate
FROM json.`/Volumes/workspace/default/data/schema/people_2.json` 

In [0]:
select * from people

#####4. Automatic schema evolution - New column in the middle
For INSERT 
1. Either it doesn't work because of the column matching by position
2. Or it corrupts your data

In [0]:
INSERT INTO people
SELECT id, fname firstName, lname lastName, phone phoneNumber, dob birthDate
FROM json.`/Volumes/workspace/default/data/schema/people_2.json`

#####5. Automatic schema evolution - New column in the middle
Works with MERGE INSERT

In [0]:
MERGE INTO people T
USING 
(
    SELECT id, fname firstName, lname lastName, phone phoneNumber, dob birthDate 
    FROM json.`/Volumes/workspace/default/data/schema/people_3.json`
) S
ON T.id = S.id
WHEN NOT MATCHED THEN 
    INSERT *

In [0]:
select * from people

####Automatic Schema Evolution at Table level

In [0]:
DROP TABLE IF EXISTS people;

CREATE OR REPLACE TABLE people(
  id INT,
  firstName STRING,
  lastName STRING
) USING DELTA;

INSERT INTO people
SELECT id, fname, lname FROM json.`/Volumes/workspace/default/data/schema/people.json`;

SELECT * FROM people;

#####6. Schema evolution - New column at the end

In [0]:
%python
from pyspark.sql.functions import to_date

people_2_schema = "id INT, fname STRING, lname STRING, dob STRING"

people_2_df =  (
      spark
      .read
      .format("json")
      .schema(people_2_schema)
      .load("/Volumes/workspace/default/data/schema/people_2.json")
      .toDF("id", "firstName", "lastName", "birthDate")
)

(
     people_2_df
      .write
      .format("delta")
      .mode("append")
      .option("mergeSchema", "true")
      .saveAsTable("people")
)

In [0]:
select * from people

#####5. Automatic schema evolution - New column in the middle

In [0]:
%python
from pyspark.sql.functions import to_date

people_3_schema = "id INT, fname STRING, lname STRING, phone STRING, dob STRING"

people_3_df =  (
      spark
      .read
      .format("json")
      .schema(people_3_schema)
      .load("/Volumes/workspace/default/data/schema/people_3.json")
      .toDF("id", "firstName", "lastName", "phoneNumber", "birthDate")
)

(
   people_3_df
      .write
      .format("delta")
      .mode("append")
      .option("mergeSchema", "true")
      .saveAsTable("people")
)

In [0]:
select * from people

###Cleanup

In [0]:
DROP DATABASE IF EXISTS demodb CASCADE